In [24]:
import importlib
import config as cfg
importlib.reload(cfg)
print(cfg.AGGREGATION_METHODS.get('culture'))

('weighted_median', {'F307': 3, 'F303': 3, 'F315': 2, 'F305': 1})


In [34]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import sys
sys.path.insert(0, r"C:\Users\admin\work\idf_project\notebooks")
import config as cfg

engine = create_engine(cfg.DB_URL)

print("Connected. Building accessibility scores...")
print(f"Dimensions to score: {list(cfg.INDEX_DIMENSIONS.keys())}")
print(f"Total indicators: {len(cfg.ALL_INDICATORS)}")

Connected. Building accessibility scores...
Dimensions to score: ['food_access', 'health_core', 'health_complementary', 'education_primary', 'education_secondary', 'education_higher', 'social_services', 'transport', 'sports', 'culture', 'public_services']
Total indicators: 35


In [35]:
# Load the full accessibility grid into pandas
# This is our base data — one row per commune × equipment type
access = pd.read_sql(
    "SELECT insee_com, typeeq_id, avg_travel_min FROM idf.accessibility_grid",
    engine
)

print(f"Loaded: {access.shape}")
print(f"Communes: {access['insee_com'].nunique()}")
print(f"Equipment types: {access['typeeq_id'].nunique()}")

Loaded: (76020, 3)
Communes: 1267
Equipment types: 60


In [36]:
def weighted_median(values, weights):
    """
    Compute weighted median.
    
    Simple explanation: sort values, accumulate weights, 
    find the value where cumulative weight crosses 50%.
    
    Technical: standard weighted median algorithm.
    Returns the value v where sum(weights where value <= v) >= total_weight/2
    """
    # Sort by value
    sorted_pairs = sorted(zip(values, weights), key=lambda x: x[0])
    vals = [p[0] for p in sorted_pairs]
    wts  = [p[1] for p in sorted_pairs]
    
    total = sum(wts)
    cumulative = 0
    for v, w in zip(vals, wts):
        cumulative += w
        if cumulative >= total / 2:
            return v
    return vals[-1]


def compute_indicator_score(access_df, indicator_name):
    """
    For a given indicator, compute one travel time per commune
    using the correct aggregation method from config.
    
    Returns a Series indexed by insee_com with travel time values.
    """
    # Get the TYPEQU codes for this indicator
    codes = [k for k, v in cfg.FACILITY_TYPES.items() if v == indicator_name]
    
    if not codes:
        print(f"Warning: no codes found for indicator {indicator_name}")
        return None
    
    # Filter access data to relevant codes
    subset = access_df[access_df['typeeq_id'].isin(codes)].copy()
    
    if subset.empty:
        print(f"Warning: no data found for indicator {indicator_name}")
        return None
    
    # Get aggregation method
    method_info = cfg.AGGREGATION_METHODS.get(indicator_name, ('min', None))
    method, weights_map = method_info
    
    if len(codes) == 1 or method == 'min':
        # Single code or MIN — take minimum travel time across codes per commune
        result = subset.groupby('insee_com')['avg_travel_min'].min()
        
    elif method == 'median':
        # Simple median across codes per commune
        result = subset.groupby('insee_com')['avg_travel_min'].median()
        
    elif method == 'weighted_median':
        # Weighted median — weight each code by its importance
        # weights_map is {typequ_code: weight}
        subset = subset.copy()
        subset['weight'] = subset['typeeq_id'].map(weights_map).fillna(1)
        
        result = subset.groupby('insee_com').apply(
            lambda g: weighted_median(
                g['avg_travel_min'].tolist(),
                g['weight'].tolist()
            )
        )
    else:
        result = subset.groupby('insee_com')['avg_travel_min'].min()
    
    return result


# Compute travel times for all indicators
print("Computing indicator travel times...")
indicator_times = {}

for indicator in cfg.ALL_INDICATORS:
    times = compute_indicator_score(access, indicator)
    if times is not None:
        indicator_times[indicator] = times
        
print(f"Computed travel times for {len(indicator_times)} indicators")

# Build a wide DataFrame: one row per commune, one column per indicator (travel time)
times_df = pd.DataFrame(indicator_times)
times_df.index.name = 'insee_com'
times_df = times_df.reset_index()

print(f"\nTravel times shape: {times_df.shape}")
print(times_df.head(3))

Computing indicator travel times...
Computed travel times for 35 indicators

Travel times shape: (1267, 36)
  insee_com  elderly_care    college  sport_pool  train_national  sport_other  \
0     75056      1.820482   1.732714    2.701899        7.577478     2.202935   
1     77001      7.069324   8.824300   18.443300       43.685591     5.897328   
2     77002      2.644369  12.423099   13.595045       40.702996    12.252713   

   higher_private  health_msp  boulangerie  sport_indoor  ...   security  \
0        5.996240    3.837764     1.076762      1.691219  ...   2.731555   
1       31.121008   17.826053     5.208271      7.718385  ...   7.087668   
2       44.751579   13.742983     9.624639     12.895290  ...  11.869111   

   train_local  sport_outdoor  health_dental  local_food_shop  lycee_pro  \
0    23.322933       1.817653       1.294440         1.081546   3.380339   
1    16.679142       2.404206       7.327486         5.513951  26.144745   
2    25.806005       2.876082     

In [37]:
def normalize_indicator(series):
    """
    Percentile-based normalization.
    Rank communes by travel time (ascending = better access).
    Scale ranks to 0-1 where 1 = best access, 0 = worst.
    
    Why: outlier-resistant, guarantees uniform score distribution,
    makes communes comparable on a consistent relative scale.
    """
    n = len(series.dropna())
    if n <= 1:
        return pd.Series(1.0, index=series.index)
    
    # Rank ascending (lower travel time = better = higher rank)
    ranks = series.rank(method='average', ascending=True)
    
    # Scale to 0-1, inverted so higher score = better access
    return 1 - (ranks - 1) / (n - 1)


# Apply normalization to every indicator column
indicator_cols = [c for c in times_df.columns if c != 'insee_com']
scores_df = times_df.copy()

for col in indicator_cols:
    scores_df[col] = normalize_indicator(times_df[col])

print("Normalization complete.")
print("\nScore statistics (should all be 0-1):")
print(scores_df[indicator_cols].describe().round(3).to_string())

Normalization complete.

Score statistics (should all be 0-1):
       elderly_care   college  sport_pool  train_national  sport_other  higher_private  health_msp  boulangerie  sport_indoor  higher_public   culture  childcare    postal  childcare_loisir  supermarket_large  health_pmi  school_primary  health_nurse  vocational_training  health_urgences  lycee_general  health_maternity  health_centre  train_regional  health_pharmacy  security  train_local  sport_outdoor  health_dental  local_food_shop  lycee_pro  social_centre  school_maternelle  health_lab  health_gp
count      1267.000  1267.000    1267.000        1267.000     1267.000        1267.000    1267.000     1267.000      1267.000       1267.000  1267.000   1267.000  1267.000          1267.000           1267.000    1267.000        1267.000      1267.000             1267.000         1267.000       1267.000          1267.000       1267.000        1267.000         1267.000  1267.000     1267.000       1267.000       1267.000       

In [38]:
def compute_dimension_score(scores_df, dimension_name, dimension_config):
    """
    Aggregate indicator scores into one dimension score per commune.
    
    Uses indicator_weights if defined, otherwise equal weights.
    """
    indicators = dimension_config['indicators']
    indicator_weights = dimension_config['indicator_weights']
    
    # Only keep indicators that exist in our scores dataframe
    available = [i for i in indicators if i in scores_df.columns]
    missing = [i for i in indicators if i not in scores_df.columns]
    
    if missing:
        print(f"  Warning: {dimension_name} missing indicators: {missing}")
    
    if not available:
        print(f"  Error: no indicators available for {dimension_name}")
        return None
    
    if indicator_weights:
        # Weighted average across indicators
        weighted_sum = sum(
            scores_df[ind] * indicator_weights[ind]
            for ind in available
            if ind in indicator_weights
        )
        total_weight = sum(
            indicator_weights[ind]
            for ind in available
            if ind in indicator_weights
        )
        return weighted_sum / total_weight
    else:
        # Simple average across indicators
        return scores_df[available].mean(axis=1)


# Compute all dimension scores
print("Computing dimension scores...")
dimension_scores = scores_df[['insee_com']].copy()

for dim_name, dim_config in cfg.INDEX_DIMENSIONS.items():
    dim_score = compute_dimension_score(scores_df, dim_name, dim_config)
    if dim_score is not None:
        dimension_scores[f'score_{dim_name}'] = dim_score.values
        print(f"  {dim_name}: avg={dim_score.mean():.3f}, "
              f"min={dim_score.min():.3f}, max={dim_score.max():.3f}")

print(f"\nDimension scores shape: {dimension_scores.shape}")

Computing dimension scores...
  food_access: avg=0.500, min=0.007, max=0.995
  health_core: avg=0.500, min=0.002, max=0.993
  health_complementary: avg=0.500, min=0.017, max=0.983
  education_primary: avg=0.500, min=0.008, max=0.987
  education_secondary: avg=0.500, min=0.009, max=0.997
  education_higher: avg=0.500, min=0.001, max=0.999
  social_services: avg=0.500, min=0.016, max=0.984
  transport: avg=0.500, min=0.025, max=0.979
  sports: avg=0.500, min=0.013, max=0.986
  culture: avg=0.500, min=0.000, max=1.000
  public_services: avg=0.500, min=0.001, max=0.991

Dimension scores shape: (1267, 12)


In [39]:
# Weighted average of dimension scores
print("Computing EAI composite score...")

dimension_cols = [c for c in dimension_scores.columns if c.startswith('score_')]
total_weight = sum(cfg.INDEX_DIMENSIONS[d.replace('score_', '')]['weight'] 
                   for d in dimension_cols)

weighted_sum = sum(
    dimension_scores[dim_col] * cfg.INDEX_DIMENSIONS[dim_col.replace('score_', '')]['weight']
    for dim_col in dimension_cols
)

dimension_scores['eai_score'] = weighted_sum / total_weight

# Add rank (1 = best equipped commune in IDF)
dimension_scores['eai_rank'] = dimension_scores['eai_score']\
    .rank(ascending=False, method='min').astype(int)

print(f"\nEAI Score statistics:")
print(dimension_scores['eai_score'].describe().round(3))
print(f"\nTop 10 best equipped communes:")

# Join commune names for readability
commune_names = pd.read_sql(
    "SELECT insee_com, nom_com FROM idf.communes", engine
)
top10 = dimension_scores.merge(commune_names, on='insee_com')\
    .nsmallest(10, 'eai_rank')[['nom_com', 'eai_score', 'eai_rank']]
print(top10.to_string())

print(f"\nBottom 10 least equipped communes:")
bottom10 = dimension_scores.merge(commune_names, on='insee_com')\
    .nlargest(10, 'eai_rank')[['nom_com', 'eai_score', 'eai_rank']]
print(bottom10.to_string())

Computing EAI composite score...

EAI Score statistics:
count    1267.000
mean        0.500
std         0.216
min         0.030
25%         0.334
50%         0.485
75%         0.674
max         0.951
Name: eai_score, dtype: float64

Top 10 best equipped communes:
                   nom_com  eai_score  eai_rank
983              Montrouge   0.951074         1
1047         Choisy-le-Roi   0.937317         2
1042                Cachan   0.931078         3
971                 Clichy   0.928831         4
1053        Ivry-sur-Seine   0.927350         5
979       Levallois-Perret   0.922558         6
1006          La Courneuve   0.922413         7
1028  Saint-Ouen-sur-Seine   0.919216         8
1024             Le Raincy   0.918955         9
997          Aubervilliers   0.918285        10

Bottom 10 least equipped communes:
                     nom_com  eai_score  eai_rank
403  Saint-Martin-du-Boschet   0.029989      1267
263               Les Marêts   0.049484      1266
423        Sancy-lès-P

In [40]:
# First truncate the existing commune_index table
with engine.connect() as conn:
    conn.execute(text("TRUNCATE TABLE idf.commune_index"))
    conn.commit()

# Get commune geometries to attach to the index
communes_geo = pd.read_sql(
    "SELECT insee_com, nom_com FROM idf.communes", engine
)

# Merge scores with commune info
final_index = communes_geo.merge(dimension_scores, on='insee_com', how='left')

print(f"Final index shape: {final_index.shape}")
print(f"Communes with scores: {final_index['eai_score'].notna().sum()}")
print(f"Communes without scores: {final_index['eai_score'].isna().sum()}")

# Write to PostGIS — non-spatial first, geometry handled separately
cols_to_write = ['insee_com', 'nom_com', 'eai_score', 'eai_rank'] + \
                [c for c in final_index.columns if c.startswith('score_')]

final_index[cols_to_write].to_sql(
    'commune_index',
    engine,
    schema='idf',
    if_exists='append',
    index=False
)

# Update geometry from communes table
with engine.connect() as conn:
    conn.execute(text("""
        UPDATE idf.commune_index ci
        SET geom = c.geom
        FROM idf.communes c
        WHERE ci.insee_com = c.insee_com
    """))
    conn.commit()
    
print("Scores written to idf.commune_index")

# Final verification
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT 
            COUNT(*) as total,
            COUNT(eai_score) as with_score,
            ROUND(AVG(eai_score)::numeric, 3) as avg_eai,
            ROUND(MIN(eai_score)::numeric, 3) as min_eai,
            ROUND(MAX(eai_score)::numeric, 3) as max_eai
        FROM idf.commune_index
    """))
    row = result.fetchone()
    print(f"\nFinal verification:")
    print(f"Total communes: {row[0]}")
    print(f"With EAI score: {row[1]}")
    print(f"Average EAI: {row[2]}")
    print(f"Min EAI: {row[3]}")
    print(f"Max EAI: {row[4]}")

Final index shape: (1266, 15)
Communes with scores: 1266
Communes without scores: 0
Scores written to idf.commune_index

Final verification:
Total communes: 1266
With EAI score: 1266
Average EAI: 0.500
Min EAI: 0.030
Max EAI: 0.951


In [41]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT 
            COUNT(*) as total,
            COUNT(eai_score) as with_score,
            COUNT(geom) as with_geometry,
            ROUND(AVG(eai_score)::numeric, 3) as avg_eai
        FROM idf.commune_index
    """))
    row = result.fetchone()
    print(f"Total rows: {row[0]}")
    print(f"With EAI score: {row[1]}")
    print(f"With geometry: {row[2]}")
    print(f"Average EAI: {row[3]}")

Total rows: 1266
With EAI score: 1266
With geometry: 1266
Average EAI: 0.500


In [42]:
# Pull culture indicator travel times per code per commune
culture_codes = ['F303', 'F305', 'F307', 'F315']
culture_labels = {
    'F303': 'cinema',
    'F305': 'conservatoire', 
    'F307': 'bibliotheque',
    'F315': 'arts_spectacle'
}

culture_data = pd.read_sql("""
    SELECT insee_com, typeeq_id, avg_travel_min
    FROM idf.accessibility_grid
    WHERE typeeq_id IN ('F303', 'F305', 'F307', 'F315')
""", engine)

# Pivot to wide format
culture_wide = culture_data.pivot(
    index='insee_com', 
    columns='typeeq_id', 
    values='avg_travel_min'
).rename(columns=culture_labels)

print("=== Culture travel time statistics ===")
print(culture_wide.describe().round(2).to_string())

print("\n=== Correlation between culture types ===")
print(culture_wide.corr().round(3).to_string())

print("\n=== How many communes have travel time > 20 min per type ===")
for col in culture_wide.columns:
    pct = (culture_wide[col] > 20).mean() * 100
    print(f"  {col}: {pct:.1f}% of communes > 20 min")

print("\n=== Current median vs per-type medians ===")
# What the current median produces
culture_wide['current_median'] = culture_wide.median(axis=1)
print(f"  Current median score range: {culture_wide['current_median'].min():.2f} - {culture_wide['current_median'].max():.2f}")
print(f"  Current median avg: {culture_wide['current_median'].mean():.2f} min")

# Per type
for col in ['cinema', 'conservatoire', 'bibliotheque', 'arts_spectacle']:
    print(f"  {col} avg: {culture_wide[col].mean():.2f} min")

=== Culture travel time statistics ===
typeeq_id   cinema  conservatoire  bibliotheque  arts_spectacle
count      1267.00        1267.00       1267.00         1267.00
mean         11.04          13.99          3.89           12.89
std           6.53           7.99          2.58            8.22
min           1.36           1.88          1.03            1.53
25%           5.43           7.86          2.13            5.77
50%          10.09          12.97          2.85           11.63
75%          15.53          18.64          5.11           18.61
max          30.02          42.96         18.28           40.12

=== Correlation between culture types ===
typeeq_id       cinema  conservatoire  bibliotheque  arts_spectacle
typeeq_id                                                          
cinema           1.000          0.551         0.386           0.638
conservatoire    0.551          1.000         0.461           0.615
bibliotheque     0.386          0.461         1.000           0.402
ar

In [43]:
def weighted_median(values, weights):
    sorted_pairs = sorted(zip(values, weights), key=lambda x: x[0])
    vals = [p[0] for p in sorted_pairs]
    wts  = [p[1] for p in sorted_pairs]
    total = sum(wts)
    cumulative = 0
    for v, w in zip(vals, wts):
        cumulative += w
        if cumulative >= total / 2:
            return v
    return vals[-1]

culture_weights = {'F307': 3, 'F303': 3, 'F315': 2, 'F305': 1}

# Compute both methods for comparison
culture_data['weight'] = culture_data['typeeq_id'].map(culture_weights)

current_median = culture_data.groupby('insee_com')['avg_travel_min'].median()

weighted_med = culture_data.groupby('insee_com').apply(
    lambda g: weighted_median(
        g['avg_travel_min'].tolist(),
        g['weight'].tolist()
    )
)

comparison = pd.DataFrame({
    'current_median': current_median,
    'weighted_median': weighted_med
})

comparison['difference'] = comparison['weighted_median'] - comparison['current_median']

print("=== Impact of switching to weighted median for culture ===")
print(f"\nCurrent median:  avg={current_median.mean():.2f}, "
      f"std={current_median.std():.2f}")
print(f"Weighted median: avg={weighted_med.mean():.2f}, "
      f"std={weighted_med.std():.2f}")

print(f"\nDifference (weighted - current):")
print(comparison['difference'].describe().round(2))

print(f"\nCommunes where weighted median is >3 min LOWER (benefits from change):")
benefiting = comparison[comparison['difference'] < -3]
print(f"  {len(benefiting)} communes ({len(benefiting)/len(comparison)*100:.1f}%)")

print(f"\nCommunes where weighted median is >3 min HIGHER (penalized by change):")
penalized = comparison[comparison['difference'] > 3]
print(f"  {len(penalized)} communes ({len(penalized)/len(comparison)*100:.1f}%)")

# Show most impacted communes
commune_names = pd.read_sql("SELECT insee_com, nom_com FROM idf.communes", engine)
comparison_named = comparison.merge(commune_names, on='insee_com')

print("\nTop 10 communes benefiting most from weighted median:")
print(comparison_named.nsmallest(10, 'difference')[
    ['nom_com', 'current_median', 'weighted_median', 'difference']
].round(2).to_string())

=== Impact of switching to weighted median for culture ===

Current median:  avg=10.81, std=5.99
Weighted median: avg=9.90, std=5.88

Difference (weighted - current):
count    1267.00
mean       -0.91
std         2.25
min       -12.39
25%        -1.32
50%        -0.36
75%        -0.02
max         9.57
Name: difference, dtype: float64

Communes where weighted median is >3 min LOWER (benefits from change):
  150 communes (11.8%)

Communes where weighted median is >3 min HIGHER (penalized by change):
  25 communes (2.0%)

Top 10 communes benefiting most from weighted median:
                       nom_com  current_median  weighted_median  difference
291                  Montenils           19.56             7.17      -12.39
313                     Nangis           13.60             2.70      -10.90
113              Clos-Fontaine           18.44             8.03      -10.41
301                 Montolivet           22.14            12.17       -9.97
362                     Quiers           

In [44]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT 
            COUNT(*) as total,
            ROUND(AVG(eai_score)::numeric, 3) as avg_eai,
            ROUND(MIN(eai_score)::numeric, 3) as min_eai,
            ROUND(MAX(eai_score)::numeric, 3) as max_eai,
            ROUND(STDDEV(eai_score)::numeric, 3) as std_eai
        FROM idf.commune_index
    """))
    row = result.fetchone()
    print(f"Total: {row[0]}")
    print(f"Average EAI: {row[1]}")
    print(f"Min EAI: {row[2]}")
    print(f"Max EAI: {row[3]}")
    print(f"Std dev: {row[4]}")

Total: 1266
Average EAI: 0.500
Min EAI: 0.030
Max EAI: 0.951
Std dev: 0.216
